In [43]:
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import random_split
from torch import device

device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available() 
    else "cpu"
)
print(f"Using device: {device}")

Using device: cpu


In [29]:
class patientRecords(Dataset):
    def __init__(self, fileName):
        # Catagorical mappings
        smoke_map = {'never': 0, 'former': 1, 'current': 2}
        sex_map = {'F': 0, 'M': 1}
        exercise_map = {'sedentary': 0, 'light': 1, 'moderate': 2, 'vigorous': 3}
        coverage_map = {'commercial': 0, 'tricare': 1, 'medicaid': 2, 'medicare': 3, 'uninsured': 4}
        alcohol_map = {'none': 0, 'light': 1, 'moderate': 2, 'heavy': 3}

        self.df = pd.read_csv(fileName)

        # Convert categorical variables to numerical using the mappings
        self.df['smoking_status'] = self.df['smoking_status'].map(smoke_map)
        self.df['sex'] = self.df['sex'].map(sex_map)
        self.df['exercise_level'] = self.df['exercise_level'].map(exercise_map)
        self.df['insurance_type'] = self.df['insurance_type'].map(coverage_map)
        self.df['alcohol_use'] = self.df['alcohol_use'].map(alcohol_map)

        # Normalize numerical features
        self.df['age'] = (self.df['age'] - self.df['age'].mean()) / self.df['age'].std()
        self.df['bmi'] = (self.df['bmi'] - self.df['bmi'].mean()) / self.df['bmi'].std()
        self.df['systolic_bp'] = (self.df['systolic_bp'] - self.df['systolic_bp'].mean()) / self.df['systolic_bp'].std()
        self.df['diastolic_bp'] = (self.df['diastolic_bp'] - self.df['diastolic_bp'].mean()) / self.df['diastolic_bp'].std()
        self.df['heart_rate'] = (self.df['heart_rate'] - self.df['heart_rate'].mean()) / self.df['heart_rate'].std()
        self.df['temperature_f'] = (self.df['temperature_f'] - self.df['temperature_f'].mean()) / self.df['temperature_f'].std()
        self.df['charlson_index'] = (self.df['charlson_index'] - self.df['charlson_index'].mean()) / self.df['charlson_index'].std()

        # Convert to tensors
        cat_data = self.df[['sex', 'smoking_status', 'alcohol_use', 'exercise_level', 'insurance_type']].values
        self.cat_tensor = torch.tensor(cat_data, dtype=torch.long)

        cont_data = self.df[['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'temperature_f', 'charlson_index']].values
        self.cont_tensor = torch.tensor(cont_data, dtype=torch.float32)

        label_data = self.df[["dx_hypertension", 'dx_type2_diabetes', 'dx_hyperlipidemia', 'dx_obesity', 'dx_coronary_artery_disease', 'dx_heart_failure', 'dx_atrial_fibrillation', 'dx_chronic_kidney_disease', 'dx_copd', 'dx_asthma', 'dx_depression', 'dx_anxiety', 'dx_hypothyroidism', 'dx_osteoarthritis', 'dx_type1_diabetes']].values
        self.label_tensor = torch.tensor(label_data, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.cat_tensor[idx], self.cont_tensor[idx], self.label_tensor[idx]
    def get_disease_names(self):
        return self.df.columns[13:].tolist()

In [74]:
class doctorMLP(nn.Module):
    def __init__(self):
        super(doctorMLP, self).__init__()

        sex_dim = 5
        smoking_dim = 5
        alcohol_dim = 5
        exercise_dim = 5
        insurance_dim = 7

        self.emb_sex = nn.Embedding(2, sex_dim)
        self.emb_smoking = nn.Embedding(3, smoking_dim)
        self.emb_alcohol = nn.Embedding(4, alcohol_dim)
        self.emb_exercise = nn.Embedding(4, exercise_dim)
        self.emb_insurance = nn.Embedding(5, insurance_dim)

        self.embeddings = nn.ModuleList([self.emb_sex, self.emb_smoking, self.emb_alcohol, self.emb_exercise, self.emb_insurance])

        layer1_input_dim = sex_dim + smoking_dim + alcohol_dim + exercise_dim + insurance_dim + 7
        self.fc1 = nn.Linear(layer1_input_dim, 128)
        self.dp1 = nn.Dropout(0.1)
        self.bn1 = nn.BatchNorm1d(128)
        torch.nn.init.kaiming_normal_(self.fc1.weight, nonlinearity='relu')
        torch.nn.init.constant_(self.fc1.bias, 0)

        self.fc2 = nn.Linear(128, 80)
        self.dp2 = nn.Dropout(0.1)
        self.bn2 = nn.BatchNorm1d(80)
        torch.nn.init.kaiming_normal_(self.fc2.weight, nonlinearity='relu')
        torch.nn.init.constant_(self.fc2.bias, 0)

        self.output = nn.Linear(80, 15)
        self.relu = nn.ReLU()

    def forward(self, cat_data, cont_data):
        embList = []
        for (i, enum) in enumerate(self.embeddings):
            embList.append(enum(cat_data[:, i]))
        cat_embedded = torch.cat(embList, dim=1)

        x = torch.cat((cat_embedded, cont_data), dim=1)

        x = self.relu(self.bn1(self.dp1(self.fc1(x))))
        x = self.relu(self.bn2(self.dp2(self.fc2(x))))
        return self.output(x)

In [72]:
# lets test

hrm = patientRecords('archive/patients.csv')
print(hrm.__getitem__(0))
print(hrm.get_disease_names())

docta = doctorMLP()
fake_cats = torch.randint(0, 2, (3, 5))
fake_conts = torch.randn(3, 7)

try:
    output = docta(fake_cats, fake_conts)
    print(f"Input Cat Shape:  {fake_cats.shape}")
    print(f"Input Cont Shape: {fake_conts.shape}")
    print(f"--- FUSION SUCCESSFUL ---")
    print(f"Output Shape:     {output.shape}") # Should be [3, 15]
except Exception as e:
    print(f"Shape Check Failed! Error: {e}")


(tensor([1, 1, 1, 2, 3]), tensor([-0.3258, -0.7428,  0.7454, -0.2719, -0.6749, -0.4069,  1.1554]), tensor([0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0.]))
['dx_hypertension', 'dx_type2_diabetes', 'dx_hyperlipidemia', 'dx_obesity', 'dx_coronary_artery_disease', 'dx_heart_failure', 'dx_atrial_fibrillation', 'dx_chronic_kidney_disease', 'dx_copd', 'dx_asthma', 'dx_depression', 'dx_anxiety', 'dx_hypothyroidism', 'dx_osteoarthritis', 'dx_type1_diabetes']
Input Cat Shape:  torch.Size([3, 5])
Input Cont Shape: torch.Size([3, 7])
--- FUSION SUCCESSFUL ---
Output Shape:     torch.Size([3, 15])


In [77]:
# training
# If deciding to use num_workers > 0, use __name__ == "__main__":

full_ds = patientRecords('archive/patients.csv')
train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_ds, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size = 1024, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size = 1024, shuffle=False, num_workers=0)

train_mean = full_ds.cont_tensor[:train_size].mean(dim=0)
train_std = full_ds.cont_tensor[:train_size].std(dim=0)

# Weighting loss--more stress of False Negatives
labels_df = full_ds.df.iloc[:, -15:]
positives = labels_df.sum()
negatives = len(labels_df) - positives
weights = np.sqrt(negatives / (positives + 1e-8)).values
# weights = (negatives / (positives + 1e-8)).values
# weights = 10
pos_weight_tensor = torch.tensor(weights, dtype=torch.float32).to(device)

mistaDocta = doctorMLP()
mistaDocta.to(device)
loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
# Weight decay punishes large weights
# optimizer = torch.optim.Adam(mistaDocta.parameters(), lr=0.0007, weight_decay=1e-5)
optimizer = torch.optim.Adam(mistaDocta.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

num_epochs = 40
threshold = 0.3

for epoch in range(num_epochs):
    mistaDocta.train()
    total_loss = 0

    for cats, conts, labels in train_loader:
        cats, conts, labels = cats.to(device), conts.to(device), labels.to(device)
        optimizer.zero_grad()

        #forward pass
        output = mistaDocta(cats, conts)

        loss_val = loss(output, labels)

        # backward pass
        loss_val.backward()
        optimizer.step()

        total_loss += loss_val.item()
        
    mistaDocta.eval()

    val_loss = 0
    completely_correct = 0
    total_correct_diseases = 0
    total_diseases = 0
    total_tp = torch.zeros(15).to(device)
    total_fp = torch.zeros(15).to(device)
    total_fn = torch.zeros(15).to(device)

    with torch.no_grad():
        for cats, conts, labels in val_loader:
            cats, conts, labels = cats.to(device), conts.to(device), labels.to(device)

            # Loss Calc
            output = mistaDocta(cats, conts)
            loss_val = loss(output, labels)
            val_loss += loss_val.item()
            
            # More sensitivity for preds bc some diseases have very low prevalence
            preds = (torch.sigmoid(output) > threshold).float()

            # Accuracy Calc
            total_correct_diseases += (preds == labels).sum().item()
            total_diseases += labels.sum().item()
            completely_correct += (preds == labels).all(dim=1).sum().item()

            # Precision/Recall Calc
            total_tp += ((preds == 1) & (labels == 1)).sum(dim=0)
            total_fp += ((preds == 1) & (labels == 0)).sum(dim=0)
            total_fn += ((preds == 0) & (labels == 1)).sum(dim=0)
        
        prescision = total_tp / (total_tp + total_fp + 1e-8)
        recall = total_tp / (total_tp + total_fn + 1e-8)
        f1_score = 2 * (prescision * recall) / (prescision + recall + 1e-8)

        macro_precision = prescision.mean().item()
        macro_recall = recall.mean().item()
        macro_f1_score = f1_score.mean().item()

    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}")
    print(f"Accuracy: Patient Acc: {completely_correct/val_size:.4f}, Disease Acc: {total_correct_diseases/total_diseases:.4f}")
    print(f"Precision: {macro_precision:.4f}, Recall: {macro_recall:.4f}, F1 Score: {macro_f1_score:.4f}")
    if epoch == num_epochs - 1:
        results_df = pd.DataFrame({
            'Disease': full_ds.get_disease_names(),
            'F1 Score': f1_score.detach().cpu().numpy().tolist()
        })
        results_df = results_df.sort_values(by="F1 Score", ascending=False)
        print(results_df)
torch.save(mistaDocta.state_dict(), 'mistaDocta_v1.pth')

Epoch 1, Loss: 0.6246, Val Loss: 0.5527
Accuracy: Patient Acc: 0.0269, Disease Acc: 3.4367
Precision: 0.2609, Recall: 0.5949, F1 Score: 0.3493
Epoch 2, Loss: 0.5461, Val Loss: 0.5317
Accuracy: Patient Acc: 0.0252, Disease Acc: 3.3404
Precision: 0.2711, Recall: 0.6910, F1 Score: 0.3660
Epoch 3, Loss: 0.5310, Val Loss: 0.5210
Accuracy: Patient Acc: 0.0249, Disease Acc: 3.4181
Precision: 0.2855, Recall: 0.6906, F1 Score: 0.3762
Epoch 4, Loss: 0.5229, Val Loss: 0.5149
Accuracy: Patient Acc: 0.0238, Disease Acc: 3.4372
Precision: 0.2956, Recall: 0.6848, F1 Score: 0.3784
Epoch 5, Loss: 0.5183, Val Loss: 0.5118
Accuracy: Patient Acc: 0.0254, Disease Acc: 3.4625
Precision: 0.2872, Recall: 0.6775, F1 Score: 0.3756
Epoch 6, Loss: 0.5152, Val Loss: 0.5098
Accuracy: Patient Acc: 0.0248, Disease Acc: 3.4529
Precision: 0.2880, Recall: 0.6830, F1 Score: 0.3785
Epoch 7, Loss: 0.5130, Val Loss: 0.5076
Accuracy: Patient Acc: 0.0264, Disease Acc: 3.4928
Precision: 0.2881, Recall: 0.6666, F1 Score: 0.3748

In [80]:
raw_data = pd.read_csv('archive/patients.csv')

# 2. Define your continuous columns (make sure names match your CSV)
cont_cols = ['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'temperature_f', 'charlson_index']

# 3. Get the "Human" means
human_means = raw_data[cont_cols].mean().tolist()
human_stds = raw_data[cont_cols].std().tolist()

print(f"Human Means (for your list): {human_means}")

Human Means (for your list): [71.63433, 27.513744, 133.64896, 84.39166, 72.0234, 98.603557, 0.81448]


In [81]:
import torch
import numpy as np

def predict_patient(raw_cats, raw_conts, model, h_means, h_stds, device):
    """
    Inference function for mistaDocta.
    raw_cats: List of 5 integers [sex, smoke, alcohol, exercise, insurance]
    raw_conts: List of 7 floats [age, bmi, sys_bp, dia_bp, hr, temp, charlson]
    """
    model.eval()
    with torch.no_grad():
        cat_tensor = torch.tensor(raw_cats, dtype=torch.long).unsqueeze(0).to(device)
        norm_conts = (np.array(raw_conts) - h_means) / (h_stds + 1e-8)
        cont_tensor = torch.tensor(norm_conts, dtype=torch.float32).unsqueeze(0).to(device)
        logits = model(cat_tensor, cont_tensor)
        probabilities = torch.sigmoid(logits).cpu().numpy()[0]
        
    return probabilities

In [83]:
raw_data[cont_cols].std().tolist()

[17.292068864296027,
 5.403874047478459,
 19.253868131697075,
 12.472988342766216,
 11.887677889682918,
 0.5002586834213908,
 1.0261056761019232]

In [85]:
human_means = [71.63433, 27.513744, 133.64896, 84.39166, 72.0234, 98.603557, 0.81448]
human_stds = [17.292068864296027, 5.403874047478459, 19.253868131697075, 12.472988342766216, 11.887677889682918, 0.5002586834213908, 1.0261056761019232]

# --- INPUT ---
# Cats: [Sex (0:F, 1:M), Smoking (0,1,2), Alcohol (0-3), Exercise (0-3), Insurance (0-4)]
raw_cats = [1, 0, 0, 2, 0] 

# Conts: [Age, BMI, Sys_BP, Dia_BP, HR, Temp, Charlson]
# These are initialized to the mean; change them to test the model
raw_conts = [
    48.0,  # Age
    25.0,  # BMI (Testing a higher BMI)
    133.6, # Systolic BP
    84.4,  # Diastolic BP
    72.0,  # Heart Rate
    98.6,  # Temperature (F)
    0.8    # Charlson Index
]

# --- RUNNING THE REPORT ---
probs = predict_patient(raw_cats, raw_conts, mistaDocta, 
                        np.array(human_means), np.array(human_stds), device)

disease_names = full_ds.get_disease_names()

print(f"\n{'Disease Name':<25} | {'Risk %':<8} | Status")
print("-" * 45)
for name, prob in zip(disease_names, probs):
    status = "FLAG" if prob > 0.35 else "Clear"
    print(f"{name:<25} | {prob*100:>7.1f}% | {status}")


Disease Name              | Risk %   | Status
---------------------------------------------
dx_hypertension           |    33.7% | Clear
dx_type2_diabetes         |    61.6% | FLAG
dx_hyperlipidemia         |    33.2% | Clear
dx_obesity                |    33.9% | Clear
dx_coronary_artery_disease |    44.7% | FLAG
dx_heart_failure          |    34.3% | Clear
dx_atrial_fibrillation    |    11.9% | Clear
dx_chronic_kidney_disease |     0.1% | Clear
dx_copd                   |    45.4% | FLAG
dx_asthma                 |    20.0% | Clear
dx_depression             |    19.4% | Clear
dx_anxiety                |    18.9% | Clear
dx_hypothyroidism         |    16.3% | Clear
dx_osteoarthritis         |    22.7% | Clear
dx_type1_diabetes         |     0.2% | Clear
